# Classification DPE — Performant / Passable / Energivore
## Visualisation t-SNE

**Entree :** `data_for_processing_final.xlsx` — la colonne `etiquette_dpe` contient directement
les 3 groupes cibles.

**Objectif :** ce notebook n'entraine pas de nouveau modele de classification. Il propose une
visualisation en 2 dimensions des donnees pretraitees via t-SNE (cf. cours, chapitre 10 -
reduction de dimension), afin de :
- detecter visuellement si les 3 classes DPE se separent naturellement dans l'espace des
  variables, independamment de tout modele ;
- comparer cette structure reelle avec les predictions du meilleur modele du pipeline
  (Random Forest, F1 macro 0,788 sur le test set, notebook precedent), pour voir visuellement
  ou ce modele se trompe.

**t-SNE (t-Student Neighborhood Embedding)** : technique non lineaire de reduction de dimension,
qui approche la distribution des distances entre observations par une loi de Student (cf. cours).
Contrairement a l'ACP (projection lineaire deja utilisee dans le notebook SVM), le t-SNE cherche
a preserver les voisinages locaux plutot que la variance globale, ce qui le rend souvent plus
efficace pour separer visuellement des groupes dans des donnees complexes.

**Limites a garder en tete :**
- Le t-SNE est couteux en temps de calcul (complexite quadratique en nombre d'observations) :
  un sous-echantillon stratifie est utilise plutot que l'integralite des donnees.
- Les distances et tailles de groupes sur une carte t-SNE n'ont pas d'interpretation absolue
  (seule la notion de voisinage local est preservee) : deux clusters proches sur la carte ne
  sont pas necessairement proches dans l'espace d'origine, et la taille apparente d'un cluster
  n'est pas une mesure fiable de sa dispersion reelle.
- Le resultat depend de l'hyperparametre `perplexity` et de l'initialisation aleatoire : deux
  executions peuvent produire des cartes visuellement differentes alors que la structure
  sous-jacente est la meme.

**Visualisations interactives (Plotly)** : les cartes t-SNE de ce notebook sont generees avec
Plotly plutot que matplotlib, pour permettre de zoomer et de faire tourner les nuages de points
a la souris directement dans le notebook (utile en particulier pour les projections 3D, ou les
points se chevauchent fortement sur une image statique).

**Plan du notebook :**
1. Imports et configuration
2. Chargement des donnees
3. Pretraitement (identique aux autres notebooks du pipeline)
4. Split train / test et sous-echantillonnage
5. Encodage des variables pour le t-SNE
6. Modele de reference (Gradient Boosting) pour la coloration par prediction
7. Visualisation t-SNE interactive en 2 dimensions (toutes classes)
   - 7.1 Vraie classe vs prediction
   - 7.2 Localisation des erreurs
8. Visualisation t-SNE interactive en 3 dimensions (toutes classes)
   - 8.1 Vraie classe vs prediction
   - 8.2 Localisation des erreurs
9. Zoom par paire de classes, en 2D et 3D, avec et sans affichage des erreurs
   - 9.1 Performant vs Passable
   - 9.2 Passable vs Energivore
   - 9.3 Performant vs Energivore
10. Conclusion


## 1. Imports et configuration

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import seaborn as sns
import warnings
import time

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.manifold import TSNE
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401 (necessaire pour projection='3d')
from sklearn.model_selection import train_test_split
from category_encoders import TargetEncoder

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-darkgrid")
sns.set_palette("husl")
%matplotlib inline


####################################### A PARAMETRER ################################

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Les 3 groupes metier, directement disponibles dans la colonne cible
TARGET = "etiquette_dpe"
GROUPS = ["Performant", "Passable", "Energivore"]

GROUP_COLORS = {
    "Performant": "#00923F",
    "Passable":   "#F2C12B",
    "Energivore": "#E6442E",
}

# Le t-SNE a un cout algorithmique quadratique en nombre d'observations : un sous-echantillon
# stratifie est utilise plutot que l'integralite du train set.
N_SAMPLE_TSNE = 6000

print("Imports OK")
print(f"  Cible : '{TARGET}' -> {GROUPS}")


Imports OK
  Cible : 'etiquette_dpe' -> ['Performant', 'Passable', 'Energivore']


## 2. Chargement des donnees

In [ ]:
FILE_PATH = "data/data_for_processing_final.xlsx"

df = pd.read_excel(FILE_PATH)
print(f"Dataset : {df.shape[0]:,} lignes x {df.shape[1]} colonnes")
print(f"Valeurs manquantes totales : {df.isnull().sum().sum()}")

vals = df[TARGET].unique()
print(f"\nValeurs uniques de '{TARGET}' : {sorted(vals)}")

print("\nDistribution de la cible :")
vc = df[TARGET].value_counts().reindex(GROUPS)
for grp, cnt in vc.items():
    pct = cnt / len(df) * 100
    print(f"  {grp:<12} {cnt:6,}  ({pct:5.1f}%)")


Dataset : 55,357 lignes x 25 colonnes
Valeurs manquantes totales : 9

Valeurs uniques de 'etiquette_dpe' : ['Energivore', 'Passable', 'Performant']

Distribution de la cible :
  Performant    2,303  (  4.2%)
  Passable     39,557  ( 71.5%)
  Energivore   13,497  ( 24.4%)


## 3. Pretraitement des donnees

Identique au pretraitement utilise dans les notebooks SVM, Arbre/Random Forest et Boosting :
suppression des lignes avec valeurs manquantes, suppression des colonnes quasi-constantes,
traitement des valeurs extremes, et encodage ordinal explicite des variables qui ont un ordre
naturel (qualite d'isolation, inertie, periode de construction).

In [4]:
def drop_missing(df):
    '''Supprime les lignes contenant au moins une valeur manquante (volume negligeable).'''
    n_before = len(df)
    df = df.dropna()
    print(f"Lignes supprimees pour valeurs manquantes : {n_before - len(df)}")
    return df


def suppress_quasi_const(df):
    '''Supprime les colonnes quasi-constantes (plus de 95% des observations partagent la
    meme valeur), qui n'apportent presque pas de variabilite utile.'''
    quasi_cst = [
        c for c in df.columns
        if c != TARGET
        and df[c].value_counts(normalize=True).iloc[0] > 0.95
    ]
    if quasi_cst:
        print(f"Colonnes quasi-constantes supprimees ({len(quasi_cst)}) : {quasi_cst}")
        df = df.drop(columns=quasi_cst)
    return df


def capping_outlier(df):
    '''Ecretage de hauteur_sous_plafond a 6m, et transformation log1p des colonnes a forte
    proportion de zeros (stockage ECS, production PV).'''
    if "hauteur_sous_plafond" in df.columns:
        df["hauteur_sous_plafond"] = df["hauteur_sous_plafond"].clip(upper=6.0)
    for col in ["volume_stockage_generateur_n1_ecs_n1",
                "production_electricite_pv_kwhep_par_an"]:
        if col in df.columns:
            df[col] = np.log1p(df[col])
    return df


def encoding_ordinal(df):
    '''
    Encode explicitement en entiers ordonnes les variables qui ont un ordre naturel, plutot
    que de les laisser au TargetEncoder (qui ignore cet ordre). Toute modalite non prevue dans
    l'ordre de reference est ajoutee a la fin et signalee.
    '''
    ordinal_vars = {
        'qualite_isolation_enveloppe':    ['insuffisante', 'moyenne', 'bonne', 'très bonne'],
        'qualite_isolation_murs':        ['insuffisante', 'moyenne', 'bonne', 'très bonne'],
        'qualite_isolation_menuiseries':  ['insuffisante', 'moyenne', 'bonne', 'très bonne'],
        'qualite_isolation_plancher_bas': ['insuffisante', 'moyenne', 'bonne', 'très bonne'],
        'classe_inertie_batiment':        ['Légère', 'Moyenne', 'Lourde', 'Très lourde'],
        'periode_construction':           ['ancien', 'transition', 'moderne', 'recent', 'tres_recent'],
    }

    for col, order in ordinal_vars.items():
        if col not in df.columns:
            continue
        vals_in_data = set(df[col].dropna().unique())
        unexpected = vals_in_data - set(order)
        if unexpected:
            print(f"  {col}: valeurs non prevues -> {unexpected}")
            for uv in unexpected:
                order.append(uv)

        mapping = {v: i for i, v in enumerate(order)}
        df[col + '_enc'] = df[col].map(mapping)
        print(f"  {col} -> encode ({len(order)} niveaux, {df[col + '_enc'].isnull().sum()} NaN)")

    df = df.drop(columns=list(ordinal_vars.keys()))
    return df


def classe_features(df):
    '''
    Classe les colonnes restantes en 4 groupes selon le traitement qu'elles recevront :
    - ordinal_enc_cols : encodees a l'etape precedente (suffixe '_enc'), passees telles quelles.
    - binary_cols      : variables binaires, passees telles quelles.
    - quant_cols       : variables numeriques continues.
    - qual_cols        : variables qualitatives nominales restantes, a encoder via TargetEncoder.
    '''
    ordinal_enc_cols = [c for c in df.columns if c.endswith("_enc")]
    binary_cols = [c for c in ["ventilation_posterieure_2012"] if c in df.columns]

    base_num = ["hauteur_sous_plafond", "volume_stockage_generateur_n1_ecs_n1",
                "production_electricite_pv_kwhep_par_an"]
    quant_cols = [c for c in base_num if c in df.columns]

    exclude = [TARGET] + ordinal_enc_cols + binary_cols + quant_cols
    qual_cols = [
        c for c in df.columns
        if c not in exclude
        and (df[c].dtype == object or str(df[c].dtype) == "str")
    ]

    passthrough_cols = ordinal_enc_cols + binary_cols
    all_features = quant_cols + qual_cols + passthrough_cols

    print(f"\nFeatures totales : {len(all_features)}")
    print(f"  Numeriques   ({len(quant_cols)}) : {quant_cols}")
    print(f"  Qualitatives ({len(qual_cols)})")
    print(f"  Passthrough  ({len(passthrough_cols)})")

    return all_features, quant_cols, qual_cols, passthrough_cols


def preprocessing(df):
    '''Enchaine toutes les etapes de pretraitement.'''
    df = drop_missing(df)
    df = suppress_quasi_const(df)
    df = capping_outlier(df)
    print()
    df = encoding_ordinal(df)
    all_features, quant_cols, qual_cols, passthrough_cols = classe_features(df)
    return df, all_features, quant_cols, qual_cols, passthrough_cols


In [5]:
df_data, all_features, quant_cols, qual_cols, passthrough_cols = preprocessing(df)


Lignes supprimees pour valeurs manquantes : 9
Colonnes quasi-constantes supprimees (6) : ['zone_climatique', 'classe_altitude', 'configuration_installation_chauffage_n1', 'configuration_installation_ecs_n1', 'type_installation_solaire_n1', 'production_electricite_pv_kwhep_par_an']

  qualite_isolation_enveloppe -> encode (4 niveaux, 0 NaN)
  qualite_isolation_murs -> encode (4 niveaux, 0 NaN)
  qualite_isolation_menuiseries -> encode (4 niveaux, 0 NaN)
  qualite_isolation_plancher_bas -> encode (4 niveaux, 0 NaN)
  classe_inertie_batiment -> encode (4 niveaux, 0 NaN)
  periode_construction -> encode (5 niveaux, 0 NaN)

Features totales : 18
  Numeriques   (2) : ['hauteur_sous_plafond', 'volume_stockage_generateur_n1_ecs_n1']
  Qualitatives (9)
  Passthrough  (7)


## 4. Split train / test et sous-echantillonnage

Le split train/test reste identique aux autres notebooks (80/20, stratifie), pour pouvoir
entrainer le Random Forest de reference dans les memes conditions. Le sous-echantillon utilise
pour le t-SNE est tire uniquement du train set, pour ne pas avoir besoin du test set ici.

In [6]:
X = df_data[all_features].copy()
y = df_data[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print(f"Train : {X_train.shape[0]:,}  |  Test : {X_test.shape[0]:,}")

X_train_sample, _, y_train_sample, _ = train_test_split(
    X_train, y_train,
    train_size=min(N_SAMPLE_TSNE, X_train.shape[0]),
    random_state=RANDOM_STATE,
    stratify=y_train,
)
print(f"\nSous-echantillon pour le t-SNE : {X_train_sample.shape[0]:,} lignes")
print("Distribution dans le sous-echantillon :")
for grp in GROUPS:
    cnt = (y_train_sample == grp).sum()
    print(f"  {grp:<12} {cnt:6,}  ({cnt / len(y_train_sample) * 100:.1f}%)")


Train : 44,278  |  Test : 11,070

Sous-echantillon pour le t-SNE : 6,000 lignes
Distribution dans le sous-echantillon :
  Performant      249  (4.2%)
  Passable      4,288  (71.5%)
  Energivore    1,463  (24.4%)


## 5. Encodage des variables pour le t-SNE

Le t-SNE travaille sur une matrice numerique : les variables qualitatives restantes doivent
etre encodees avant de calculer les distances entre observations. Le meme `TargetEncoder` que
dans les autres notebooks est utilise ici, fitte uniquement sur le sous-echantillon.

In [7]:
def make_preprocessor():
    '''Construit le ColumnTransformer pour preparer les donnees avant le t-SNE.'''
    cat_tr = TargetEncoder(smoothing=3.0, min_samples_leaf=50)
    transformers = [("num", "passthrough", quant_cols)]
    if qual_cols:
        transformers.append(("cat", cat_tr, qual_cols))
    if passthrough_cols:
        transformers.append(("pass", "passthrough", passthrough_cols))
    return ColumnTransformer(transformers=transformers, remainder="drop",
                              verbose_feature_names_out=False)


preprocessor_tsne = make_preprocessor()
X_train_sample_enc = preprocessor_tsne.fit_transform(X_train_sample, y_train_sample)
print(f"Donnees encodees pour le t-SNE : {X_train_sample_enc.shape}")


Donnees encodees pour le t-SNE : (6000, 18)


## 6. Modele de reference (Gradient Boosting)

Reentrainement rapide d'un Gradient Boosting, avec les hyperparametres retenus dans le notebook
`Pipeline_Best_Models` (N_TRIALS=50, meilleur modele final du pipeline, F1 macro 0.799), pour
pouvoir colorer la carte t-SNE par prediction et la comparer a la coloration par vraie classe.

**Note sur les poids de classe** : les poids exacts retenus par Optuna dans `Pipeline_Best_Models`
n'ont pas ete conserves dans la sortie de ce notebook. Les valeurs ci-dessous reprennent celles
du notebook Boosting individuel a titre d'approximation raisonnable ; ajuster si les poids exacts
sont disponibles par ailleurs.

In [8]:
'''
Reentraine un Gradient Boosting sur l'integralite du train set (et non le sous-echantillon),
afin d'obtenir des predictions de bonne qualite sur le sous-echantillon utilise pour le t-SNE.
Hyperparametres repris du notebook Pipeline_Best_Models (meilleur modele final du pipeline).
Le desequilibre des classes est gere par sample_weight (pas de class_weight disponible pour
ce modele).
'''
def make_sample_weight(y, weight_performant, weight_energivore):
    weights_map = {"Performant": weight_performant, "Passable": 1, "Energivore": weight_energivore}
    return y.map(weights_map).values


sw_train = make_sample_weight(y_train, weight_performant=8, weight_energivore=5)

preprocessor_full = make_preprocessor()
model_gb = Pipeline([
    ("pre", preprocessor_full),
    ("clf", GradientBoostingClassifier(
        n_estimators=250, learning_rate=0.0298, max_depth=7, subsample=0.501,
        random_state=RANDOM_STATE,
    )),
])
model_gb.fit(X_train, y_train, clf__sample_weight=sw_train)

y_pred_sample = model_gb.predict(X_train_sample)
print("Gradient Boosting entraine.")
print(f"Accuracy sur le sous-echantillon t-SNE : {(y_pred_sample == y_train_sample.values).mean():.3f}")


Gradient Boosting entraine.
Accuracy sur le sous-echantillon t-SNE : 0.813


## 7. Visualisation t-SNE interactive en 2 dimensions

Calcul de la projection t-SNE en 2 dimensions sur le sous-echantillon encode. La carte est
d'abord affichee simplement (vraie classe vs prediction), puis avec les erreurs du modele
superposees.

In [9]:
'''
Calcule la projection t-SNE en 2 dimensions. La perplexity (par defaut 30) controle le nombre
effectif de voisins pris en compte autour de chaque point ; elle est laissee a sa valeur par
defaut ici, adaptee a la taille du sous-echantillon utilise.
'''
from plotly.subplots import make_subplots

t0 = time.time()
tsne = TSNE(
    n_components=2,
    perplexity=30,
    init="pca",
    random_state=RANDOM_STATE,
)
X_tsne = tsne.fit_transform(X_train_sample_enc)
print(f"t-SNE 2D calcule en {time.time() - t0:.1f}s")


t-SNE 2D calcule en 19.2s


### 7.1 Vraie classe vs prediction

In [10]:
'''
Affiche, en Plotly, la projection t-SNE 2D coloree par la vraie classe DPE et par la prediction
du Gradient Boosting, sur deux panneaux cote a cote. Zoomable et survolable a la souris.
'''
from plotly.subplots import make_subplots

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Projection t-SNE 2D - vraie classe", "Projection t-SNE 2D - prediction (Gradient Boosting)"),
)

for grp in GROUPS:
    mask = y_train_sample.values == grp
    fig.add_trace(
        go.Scatter(x=X_tsne[mask, 0], y=X_tsne[mask, 1], mode="markers",
                    marker=dict(size=5, color=GROUP_COLORS[grp], opacity=0.6),
                    name=grp, legendgroup=grp, showlegend=True),
        row=1, col=1,
    )

for grp in GROUPS:
    mask = y_pred_sample == grp
    fig.add_trace(
        go.Scatter(x=X_tsne[mask, 0], y=X_tsne[mask, 1], mode="markers",
                    marker=dict(size=5, color=GROUP_COLORS[grp], opacity=0.6),
                    name=grp, legendgroup=grp, showlegend=False),
        row=1, col=2,
    )

fig.update_layout(width=1200, height=550, title_text="Projection t-SNE 2D - vraie classe vs prediction")
fig.update_xaxes(title_text="Dimension t-SNE 1")
fig.update_yaxes(title_text="Dimension t-SNE 2")
fig.show()


#### <span style="text-decoration: underline; color:red; font-weight:bold;">Interpretation :</span>

### 7.2 Localisation des erreurs

In [11]:
'''
Affiche, en Plotly, les observations mal classees par le Gradient Boosting sur la carte t-SNE
2D, pour localiser visuellement ou se concentrent les erreurs.
'''
mask_erreur = y_pred_sample != y_train_sample.values

fig = go.Figure()

for grp in GROUPS:
    mask = (y_train_sample.values == grp) & (~mask_erreur)
    fig.add_trace(go.Scatter(
        x=X_tsne[mask, 0], y=X_tsne[mask, 1], mode="markers",
        marker=dict(size=5, color=GROUP_COLORS[grp], opacity=0.35),
        name=f"{grp} (bien classe)",
    ))

fig.add_trace(go.Scatter(
    x=X_tsne[mask_erreur, 0], y=X_tsne[mask_erreur, 1], mode="markers",
    marker=dict(size=7, color="black", symbol="x", opacity=0.9),
    name="Mal classe",
))

fig.update_layout(
    title="Localisation des erreurs du Gradient Boosting sur la carte t-SNE 2D",
    xaxis_title="Dimension t-SNE 1", yaxis_title="Dimension t-SNE 2",
    width=850, height=700,
)
fig.show()

print(f"Observations mal classees dans le sous-echantillon : {mask_erreur.sum()} / {len(mask_erreur)} "
      f"({mask_erreur.mean():.1%})")


Observations mal classees dans le sous-echantillon : 1122 / 6000 (18.7%)


#### <span style="text-decoration: underline; color:red; font-weight:bold;">Interpretation :</span>

## 8. Visualisation t-SNE interactive en 3 dimensions

Meme principe que la section 7, mais en projetant sur 3 dimensions plutot que 2. La rotation a
la souris permet de retrouver une lecture correcte des distances et des chevauchements, qui
seraient masques par les occlusions sur une image 3D statique.

In [12]:
'''
Calcule la projection t-SNE en 3 dimensions, sur le meme sous-echantillon encode que la version
2D. Un nouvel objet TSNE est necessaire (n_components ne peut pas etre change apres coup).
'''
t0 = time.time()
tsne_3d = TSNE(
    n_components=3,
    perplexity=30,
    init="pca",
    random_state=RANDOM_STATE,
)
X_tsne_3d = tsne_3d.fit_transform(X_train_sample_enc)
print(f"t-SNE 3D calcule en {time.time() - t0:.1f}s")


t-SNE 3D calcule en 31.5s


### 8.1 Vraie classe vs prediction

In [13]:
'''
Affiche, en Plotly, la projection t-SNE 3D coloree par la vraie classe DPE et par la prediction
du Gradient Boosting, sur deux scenes cote a cote. Rotation et zoom a la souris.
'''
fig = make_subplots(
    rows=1, cols=2,
    specs=[[{"type": "scene"}, {"type": "scene"}]],
    subplot_titles=("Projection t-SNE 3D - vraie classe", "Projection t-SNE 3D - prediction (Gradient Boosting)"),
)

for grp in GROUPS:
    mask = y_train_sample.values == grp
    fig.add_trace(
        go.Scatter3d(x=X_tsne_3d[mask, 0], y=X_tsne_3d[mask, 1], z=X_tsne_3d[mask, 2],
                     mode="markers", marker=dict(size=3, color=GROUP_COLORS[grp], opacity=0.5),
                     name=grp, legendgroup=grp, showlegend=True),
        row=1, col=1,
    )

for grp in GROUPS:
    mask = y_pred_sample == grp
    fig.add_trace(
        go.Scatter3d(x=X_tsne_3d[mask, 0], y=X_tsne_3d[mask, 1], z=X_tsne_3d[mask, 2],
                     mode="markers", marker=dict(size=3, color=GROUP_COLORS[grp], opacity=0.5),
                     name=grp, legendgroup=grp, showlegend=False),
        row=1, col=2,
    )

fig.update_layout(width=1200, height=650, title_text="Projection t-SNE 3D - vraie classe vs prediction")
fig.show()


#### <span style="text-decoration: underline; color:red; font-weight:bold;">Interpretation :</span>

### 8.2 Localisation des erreurs

In [14]:
'''
Version interactive (Plotly) de la localisation des erreurs en 3D : permet de faire tourner
le nuage de points a la souris pour mieux distinguer les zones de chevauchement, impossibles
a lire correctement sur une image statique.
'''
mask_erreur_3d = y_pred_sample != y_train_sample.values

fig = go.Figure()

for grp in GROUPS:
    mask = (y_train_sample.values == grp) & (~mask_erreur_3d)
    fig.add_trace(go.Scatter3d(
        x=X_tsne_3d[mask, 0], y=X_tsne_3d[mask, 1], z=X_tsne_3d[mask, 2],
        mode="markers",
        marker=dict(size=3, color=GROUP_COLORS[grp], opacity=0.4),
        name=f"{grp} (bien classe)",
    ))

fig.add_trace(go.Scatter3d(
    x=X_tsne_3d[mask_erreur_3d, 0], y=X_tsne_3d[mask_erreur_3d, 1], z=X_tsne_3d[mask_erreur_3d, 2],
    mode="markers",
    marker=dict(size=1, color="black", symbol="x", opacity=0.9),
    name="Mal classe",
))

fig.update_layout(
    title="Localisation des erreurs du Gradient Boosting sur la carte t-SNE 3D",
    scene=dict(xaxis_title="Dim 1", yaxis_title="Dim 2", zaxis_title="Dim 3"),
    width=900, height=700,
    legend=dict(itemsizing="constant"),
)
fig.show()

print(f"Observations mal classees dans le sous-echantillon : {mask_erreur_3d.sum()} / {len(mask_erreur_3d)} "
      f"({mask_erreur_3d.mean():.1%})")


Observations mal classees dans le sous-echantillon : 1122 / 6000 (18.7%)


#### <span style="text-decoration: underline; color:red; font-weight:bold;">Interpretation :</span>

## 9. Zoom par paire de classes

La carte t-SNE complete (sections 7 et 8) affiche les 3 classes simultanement, ce qui peut
masquer le degre de separation reel entre deux classes prises isolement. Cette section isole
chaque paire de classes a tour de role, en 2D puis en 3D, d'abord sans puis avec les erreurs du
Gradient Boosting superposees.

**A noter** : isoler une paire ne recalcule pas le t-SNE — les coordonnees affichees sont celles
des projections completes a 3 classes (`X_tsne` et `X_tsne_3d`, sections 7 et 8), simplement
filtrees. La position relative des points reste donc influencee par la troisieme classe
masquee.

### 9.1 Performant vs Passable

**Performant vs Passable — projection 2D, sans les erreurs**

In [15]:
'''
Isole les observations Performant et Passable sur la projection t-SNE 2D (section 7), pour
visualiser plus precisement la frontiere entre ces deux classes.
'''
paire_91 = ["Performant", "Passable"]

fig = go.Figure()
for grp in paire_91:
    mask = y_train_sample.values == grp
    fig.add_trace(go.Scatter(
        x=X_tsne[mask, 0], y=X_tsne[mask, 1], mode="markers",
        marker=dict(size=6, color=GROUP_COLORS[grp], opacity=0.6),
        name=grp,
    ))

fig.update_layout(
    title="Projection t-SNE 2D - Performant vs Passable",
    xaxis_title="Dimension t-SNE 1", yaxis_title="Dimension t-SNE 2",
    width=850, height=650,
)
fig.show()


#### <span style="text-decoration: underline; color:red; font-weight:bold;">Interpretation :</span>

**Performant vs Passable — projection 2D, avec les erreurs**

In [18]:
'''
Meme zoom Performant / Passable que ci-dessus, avec superposition des observations mal
classees par le Gradient Boosting parmi ces deux classes.
'''
mask_paire_2d = np.isin(y_train_sample.values, paire_91)
mask_erreur_paire_2d = mask_paire_2d & (y_pred_sample != y_train_sample.values)

fig = go.Figure()
for grp in paire_91:
    mask = (y_train_sample.values == grp) & (~mask_erreur_paire_2d)
    fig.add_trace(go.Scatter(
        x=X_tsne[mask, 0], y=X_tsne[mask, 1], mode="markers",
        marker=dict(size=6, color=GROUP_COLORS[grp], opacity=0.4),
        name=f"{grp} (bien classe)",
    ))

fig.add_trace(go.Scatter(
    x=X_tsne[mask_erreur_paire_2d, 0], y=X_tsne[mask_erreur_paire_2d, 1], mode="markers",
    marker=dict(size=8, color="black", symbol="x", opacity=0.9),
    name="Mal classe",
))

fig.update_layout(
    title="Projection t-SNE 2D - Performant vs Passable (erreurs)",
    xaxis_title="Dimension t-SNE 1", yaxis_title="Dimension t-SNE 2",
    width=850, height=650,
)
fig.show()

print(f"Erreurs dans cette paire : {mask_erreur_paire_2d.sum()} / {mask_paire_2d.sum()} "
      f"({mask_erreur_paire_2d.sum() / mask_paire_2d.sum():.1%})")


Erreurs dans cette paire : 1056 / 4537 (23.3%)


#### <span style="text-decoration: underline; color:red; font-weight:bold;">Interpretation :</span>

**Performant vs Passable — projection 3D, sans les erreurs**

In [19]:
'''
Isole les observations Performant et Passable sur la projection t-SNE 3D (section 8).
Rotation et zoom a la souris.
'''
paire_91 = ["Performant", "Passable"]

fig = go.Figure()
for grp in paire_91:
    mask = y_train_sample.values == grp
    fig.add_trace(go.Scatter3d(
        x=X_tsne_3d[mask, 0], y=X_tsne_3d[mask, 1], z=X_tsne_3d[mask, 2], mode="markers",
        marker=dict(size=4, color=GROUP_COLORS[grp], opacity=0.6),
        name=grp,
    ))

fig.update_layout(
    title="Projection t-SNE 3D - Performant vs Passable",
    scene=dict(xaxis_title="Dim 1", yaxis_title="Dim 2", zaxis_title="Dim 3"),
    width=850, height=700,
)
fig.show()


#### <span style="text-decoration: underline; color:red; font-weight:bold;">Interpretation :</span>

**Performant vs Passable — projection 3D, avec les erreurs**

In [21]:
'''
Meme zoom Performant / Passable en 3D, avec superposition des observations mal classees par
le Gradient Boosting parmi ces deux classes.
'''
mask_paire_3d = np.isin(y_train_sample.values, paire_91)
mask_erreur_paire_3d = mask_paire_3d & (y_pred_sample != y_train_sample.values)

fig = go.Figure()
for grp in paire_91:
    mask = (y_train_sample.values == grp) & (~mask_erreur_paire_3d)
    fig.add_trace(go.Scatter3d(
        x=X_tsne_3d[mask, 0], y=X_tsne_3d[mask, 1], z=X_tsne_3d[mask, 2], mode="markers",
        marker=dict(size=4, color=GROUP_COLORS[grp], opacity=0.4),
        name=f"{grp} (bien classe)",
    ))

fig.add_trace(go.Scatter3d(
    x=X_tsne_3d[mask_erreur_paire_3d, 0], y=X_tsne_3d[mask_erreur_paire_3d, 1], z=X_tsne_3d[mask_erreur_paire_3d, 2],
    mode="markers",
    marker=dict(size=2, color="black", symbol="x", opacity=0.9),
    name="Mal classe",
))

fig.update_layout(
    title="Projection t-SNE 3D - Performant vs Passable (erreurs)",
    scene=dict(xaxis_title="Dim 1", yaxis_title="Dim 2", zaxis_title="Dim 3"),
    width=900, height=700,
)
fig.show()

print(f"Erreurs dans cette paire : {mask_erreur_paire_3d.sum()} / {mask_paire_3d.sum()} "
      f"({mask_erreur_paire_3d.sum() / mask_paire_3d.sum():.1%})")


Erreurs dans cette paire : 1056 / 4537 (23.3%)


#### <span style="text-decoration: underline; color:red; font-weight:bold;">Interpretation :</span>

### 9.2 Passable vs Energivore

**Passable vs Energivore — projection 2D, sans les erreurs**

In [22]:
'''
Isole les observations Passable et Energivore sur la projection t-SNE 2D (section 7), pour
visualiser plus precisement la frontiere entre ces deux classes.
'''
paire_92 = ["Passable", "Energivore"]

fig = go.Figure()
for grp in paire_92:
    mask = y_train_sample.values == grp
    fig.add_trace(go.Scatter(
        x=X_tsne[mask, 0], y=X_tsne[mask, 1], mode="markers",
        marker=dict(size=6, color=GROUP_COLORS[grp], opacity=0.6),
        name=grp,
    ))

fig.update_layout(
    title="Projection t-SNE 2D - Passable vs Energivore",
    xaxis_title="Dimension t-SNE 1", yaxis_title="Dimension t-SNE 2",
    width=850, height=650,
)
fig.show()


#### <span style="text-decoration: underline; color:red; font-weight:bold;">Interpretation :</span>

**Passable vs Energivore — projection 2D, avec les erreurs**

In [23]:
'''
Meme zoom Passable / Energivore que ci-dessus, avec superposition des observations mal
classees par le Gradient Boosting parmi ces deux classes.
'''
mask_paire_2d = np.isin(y_train_sample.values, paire_92)
mask_erreur_paire_2d = mask_paire_2d & (y_pred_sample != y_train_sample.values)

fig = go.Figure()
for grp in paire_92:
    mask = (y_train_sample.values == grp) & (~mask_erreur_paire_2d)
    fig.add_trace(go.Scatter(
        x=X_tsne[mask, 0], y=X_tsne[mask, 1], mode="markers",
        marker=dict(size=6, color=GROUP_COLORS[grp], opacity=0.4),
        name=f"{grp} (bien classe)",
    ))

fig.add_trace(go.Scatter(
    x=X_tsne[mask_erreur_paire_2d, 0], y=X_tsne[mask_erreur_paire_2d, 1], mode="markers",
    marker=dict(size=8, color="black", symbol="x", opacity=0.9),
    name="Mal classe",
))

fig.update_layout(
    title="Projection t-SNE 2D - Passable vs Energivore (erreurs)",
    xaxis_title="Dimension t-SNE 1", yaxis_title="Dimension t-SNE 2",
    width=850, height=650,
)
fig.show()

print(f"Erreurs dans cette paire : {mask_erreur_paire_2d.sum()} / {mask_paire_2d.sum()} "
      f"({mask_erreur_paire_2d.sum() / mask_paire_2d.sum():.1%})")


Erreurs dans cette paire : 1109 / 5751 (19.3%)


#### <span style="text-decoration: underline; color:red; font-weight:bold;">Interpretation :</span>

**Passable vs Energivore — projection 3D, sans les erreurs**

In [ ]:
'''
Isole les observations Passable et Energivore sur la projection t-SNE 3D (section 8).
Rotation et zoom a la souris.
'''
paire_92 = ["Passable", "Energivore"]

fig = go.Figure()
for grp in paire_92:
    mask = y_train_sample.values == grp
    fig.add_trace(go.Scatter3d(
        x=X_tsne_3d[mask, 0], y=X_tsne_3d[mask, 1], z=X_tsne_3d[mask, 2], mode="markers",
        marker=dict(size=4, color=GROUP_COLORS[grp], opacity=0.6),
        name=grp,
    ))

fig.update_layout(
    title="Projection t-SNE 3D - Passable vs Energivore",
    scene=dict(xaxis_title="Dim 1", yaxis_title="Dim 2", zaxis_title="Dim 3"),
    width=850, height=700,
)
fig.show()


#### <span style="text-decoration: underline; color:red; font-weight:bold;">Interpretation :</span>

**Passable vs Energivore — projection 3D, avec les erreurs**

In [24]:
'''
Meme zoom Passable / Energivore en 3D, avec superposition des observations mal classees par
le Gradient Boosting parmi ces deux classes.
'''
mask_paire_3d = np.isin(y_train_sample.values, paire_92)
mask_erreur_paire_3d = mask_paire_3d & (y_pred_sample != y_train_sample.values)

fig = go.Figure()
for grp in paire_92:
    mask = (y_train_sample.values == grp) & (~mask_erreur_paire_3d)
    fig.add_trace(go.Scatter3d(
        x=X_tsne_3d[mask, 0], y=X_tsne_3d[mask, 1], z=X_tsne_3d[mask, 2], mode="markers",
        marker=dict(size=4, color=GROUP_COLORS[grp], opacity=0.4),
        name=f"{grp} (bien classe)",
    ))

fig.add_trace(go.Scatter3d(
    x=X_tsne_3d[mask_erreur_paire_3d, 0], y=X_tsne_3d[mask_erreur_paire_3d, 1], z=X_tsne_3d[mask_erreur_paire_3d, 2],
    mode="markers",
    marker=dict(size=2, color="black", symbol="x", opacity=0.9),
    name="Mal classe",
))

fig.update_layout(
    title="Projection t-SNE 3D - Passable vs Energivore (erreurs)",
    scene=dict(xaxis_title="Dim 1", yaxis_title="Dim 2", zaxis_title="Dim 3"),
    width=900, height=700,
)
fig.show()

print(f"Erreurs dans cette paire : {mask_erreur_paire_3d.sum()} / {mask_paire_3d.sum()} "
      f"({mask_erreur_paire_3d.sum() / mask_paire_3d.sum():.1%})")


Erreurs dans cette paire : 1109 / 5751 (19.3%)


#### <span style="text-decoration: underline; color:red; font-weight:bold;">Interpretation :</span>

### 9.3 Performant vs Energivore

Ce sont les deux classes les plus eloignees en performance energetique, et celles entre lesquelles aucune confusion directe n'a ete observee dans les matrices de confusion des notebooks precedents : cette section permet de verifier si cette absence de confusion se voit aussi visuellement (separation nette) ou si les deux classes restent proches malgre tout.

**Performant vs Energivore — projection 2D, sans les erreurs**

In [25]:
'''
Isole les observations Performant et Energivore sur la projection t-SNE 2D (section 7), pour
visualiser plus precisement la frontiere entre ces deux classes.
'''
paire_93 = ["Performant", "Energivore"]

fig = go.Figure()
for grp in paire_93:
    mask = y_train_sample.values == grp
    fig.add_trace(go.Scatter(
        x=X_tsne[mask, 0], y=X_tsne[mask, 1], mode="markers",
        marker=dict(size=6, color=GROUP_COLORS[grp], opacity=0.6),
        name=grp,
    ))

fig.update_layout(
    title="Projection t-SNE 2D - Performant vs Energivore",
    xaxis_title="Dimension t-SNE 1", yaxis_title="Dimension t-SNE 2",
    width=850, height=650,
)
fig.show()


#### <span style="text-decoration: underline; color:red; font-weight:bold;">Interpretation :</span>

**Performant vs Energivore — projection 2D, avec les erreurs**

In [26]:
'''
Meme zoom Performant / Energivore que ci-dessus, avec superposition des observations mal
classees par le Gradient Boosting parmi ces deux classes.
'''
mask_paire_2d = np.isin(y_train_sample.values, paire_93)
mask_erreur_paire_2d = mask_paire_2d & (y_pred_sample != y_train_sample.values)

fig = go.Figure()
for grp in paire_93:
    mask = (y_train_sample.values == grp) & (~mask_erreur_paire_2d)
    fig.add_trace(go.Scatter(
        x=X_tsne[mask, 0], y=X_tsne[mask, 1], mode="markers",
        marker=dict(size=6, color=GROUP_COLORS[grp], opacity=0.4),
        name=f"{grp} (bien classe)",
    ))

fig.add_trace(go.Scatter(
    x=X_tsne[mask_erreur_paire_2d, 0], y=X_tsne[mask_erreur_paire_2d, 1], mode="markers",
    marker=dict(size=8, color="black", symbol="x", opacity=0.9),
    name="Mal classe",
))

fig.update_layout(
    title="Projection t-SNE 2D - Performant vs Energivore (erreurs)",
    xaxis_title="Dimension t-SNE 1", yaxis_title="Dimension t-SNE 2",
    width=850, height=650,
)
fig.show()

print(f"Erreurs dans cette paire : {mask_erreur_paire_2d.sum()} / {mask_paire_2d.sum()} "
      f"({mask_erreur_paire_2d.sum() / mask_paire_2d.sum():.1%})")


Erreurs dans cette paire : 79 / 1712 (4.6%)


#### <span style="text-decoration: underline; color:red; font-weight:bold;">Interpretation :</span>

**Performant vs Energivore — projection 3D, sans les erreurs**

In [27]:
'''
Isole les observations Performant et Energivore sur la projection t-SNE 3D (section 8).
Rotation et zoom a la souris.
'''
paire_93 = ["Performant", "Energivore"]

fig = go.Figure()
for grp in paire_93:
    mask = y_train_sample.values == grp
    fig.add_trace(go.Scatter3d(
        x=X_tsne_3d[mask, 0], y=X_tsne_3d[mask, 1], z=X_tsne_3d[mask, 2], mode="markers",
        marker=dict(size=4, color=GROUP_COLORS[grp], opacity=0.6),
        name=grp,
    ))

fig.update_layout(
    title="Projection t-SNE 3D - Performant vs Energivore",
    scene=dict(xaxis_title="Dim 1", yaxis_title="Dim 2", zaxis_title="Dim 3"),
    width=850, height=700,
)
fig.show()


#### <span style="text-decoration: underline; color:red; font-weight:bold;">Interpretation :</span>

**Performant vs Energivore — projection 3D, avec les erreurs**

In [ ]:
'''
Meme zoom Performant / Energivore en 3D, avec superposition des observations mal classees par
le Gradient Boosting parmi ces deux classes.
'''
mask_paire_3d = np.isin(y_train_sample.values, paire_93)
mask_erreur_paire_3d = mask_paire_3d & (y_pred_sample != y_train_sample.values)

fig = go.Figure()
for grp in paire_93:
    mask = (y_train_sample.values == grp) & (~mask_erreur_paire_3d)
    fig.add_trace(go.Scatter3d(
        x=X_tsne_3d[mask, 0], y=X_tsne_3d[mask, 1], z=X_tsne_3d[mask, 2], mode="markers",
        marker=dict(size=4, color=GROUP_COLORS[grp], opacity=0.4),
        name=f"{grp} (bien classe)",
    ))

fig.add_trace(go.Scatter3d(
    x=X_tsne_3d[mask_erreur_paire_3d, 0], y=X_tsne_3d[mask_erreur_paire_3d, 1], z=X_tsne_3d[mask_erreur_paire_3d, 2],
    mode="markers",
    marker=dict(size=1, color="black", symbol="x", opacity=0.9),
    name="Mal classe",
))

fig.update_layout(
    title="Projection t-SNE 3D - Performant vs Energivore (erreurs)",
    scene=dict(xaxis_title="Dim 1", yaxis_title="Dim 2", zaxis_title="Dim 3"),
    width=900, height=700,
)
fig.show()

print(f"Erreurs dans cette paire : {mask_erreur_paire_3d.sum()} / {mask_paire_3d.sum()} "
      f"({mask_erreur_paire_3d.sum() / mask_paire_3d.sum():.1%})")


Erreurs dans cette paire : 79 / 1712 (4.6%)


#### <span style="text-decoration: underline; color:red; font-weight:bold;">Interpretation :</span>

## 10. Conclusion

A completer une fois les resultats analyses : les 3 classes DPE se separent-elles visuellement
bien ou mal sur la carte t-SNE, ou se concentrent les erreurs du Gradient Boosting (frontieres
entre classes adjacentes, zones de chevauchement), et ce que cela confirme ou nuance par rapport
aux diagnostics deja obtenus dans les notebooks precedents (matrices de confusion, courbes
ROC/PR), si la projection 3D apporte une information supplementaire par rapport a la 2D, et ce
que montre le zoom par paire de classes (section 9) sur la difficulte relative a separer chaque
paire. Rappeler aussi les limites propres au t-SNE (pas de notion absolue de distance ou de
taille de cluster, dependance a la perplexity et a l'initialisation).